# Assignment: Event-Driven Backtest and Trading Engine
**Order Book Dynamics, Order Events, Fills, and Trade Management**

| Part | Focus | Methods |
|------|-------|---------|
| 1 | Order Book Dynamics | LOB simulation, depth chart, spread & impact |
| 2 | Order Events & Lifecycle | State machine, event queue, latency injection |
| 3 | Fills & Execution | Partial fills, aggressive vs passive, fill probability |
| 4 | Order Management (OMS) | Position tracking, real-time P&L, risk controls |
| 5 | Trade Management & P&L Attribution | Trade lifecycle, cost/alpha/market decomposition |

> *All data is fully simulated; no external feeds required.*

In [ ]:
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from enum import Enum, auto
from collections import deque, defaultdict
from typing import Optional, List, Dict
import plotly.graph_objects as go
import plotly.subplots as ps
import warnings
warnings.filterwarnings('ignore')

SEED   = 42
rng    = np.random.default_rng(SEED)
ANNUAL = 252

# ── Simulated mid-price process (Ornstein-Uhlenbeck mean-reverting) ───────────
T        = 2_000      # ticks
MID0     = 100.0
VOL_TICK = 0.05       # per-tick vol ($)
KAPPA    = 0.01       # mean-reversion speed

mid_prices = np.zeros(T)
mid_prices[0] = MID0
for t in range(1, T):
    mid_prices[t] = (mid_prices[t-1]
                     - KAPPA * (mid_prices[t-1] - MID0)
                     + VOL_TICK * rng.standard_normal())

tick_times = np.cumsum(rng.exponential(scale=0.5, size=T))   # irregular tick spacing (seconds)

print(f'Simulated {T} ticks | price range ${mid_prices.min():.2f} – ${mid_prices.max():.2f}')
print(f'Time span: {tick_times[-1]/60:.1f} minutes | avg tick interval: {tick_times.mean():.2f}s')

---
## Part 1 | Order Book Dynamics

A **Limit Order Book (LOB)** is the central data structure of any electronic market. At each moment it holds:
- **Bids** — buy orders sorted descending by price (best bid at top)
- **Asks** — sell orders sorted ascending by price (best ask at top)

Key quantities:

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **Spread** | $P_{\text{ask}} - P_{\text{bid}}$ | Cost of immediate execution |
| **Mid price** | $(P_{\text{ask}} + P_{\text{bid}}) / 2$ | Fair value estimate |
| **Depth** | $\sum Q_i$ at each level | Available liquidity |
| **Order Book Imbalance** | $(V_{\text{bid}} - V_{\text{ask}}) / (V_{\text{bid}} + V_{\text{ask}})$ | Short-term price pressure |

In [ ]:
# ── Simulate a 5-level LOB snapshot at each tick ──────────────────────────────
N_LEVELS     = 5
TICK_SIZE    = 0.01
BASE_SIZE    = 200    # shares at best level

def generate_lob_snapshot(mid, rng_loc, n_levels=N_LEVELS):
    """
    Generate a synthetic LOB snapshot around a given mid price.
    Returns (bid_prices, bid_sizes, ask_prices, ask_sizes)
    """
    half_spread_ticks = rng_loc.integers(1, 4)       # 1–3 tick half-spread
    best_bid = round(mid - half_spread_ticks * TICK_SIZE, 2)
    best_ask = round(mid + half_spread_ticks * TICK_SIZE, 2)

    bid_prices = np.array([round(best_bid - i * TICK_SIZE, 2) for i in range(n_levels)])
    ask_prices = np.array([round(best_ask + i * TICK_SIZE, 2) for i in range(n_levels)])

    # Size decays with distance from best; add noise
    decay   = np.exp(-0.4 * np.arange(n_levels))
    bid_sizes = (BASE_SIZE * decay * rng_loc.lognormal(0, 0.4, n_levels)).astype(int).clip(10)
    ask_sizes = (BASE_SIZE * decay * rng_loc.lognormal(0, 0.4, n_levels)).astype(int).clip(10)

    return bid_prices, bid_sizes, ask_prices, ask_sizes

# Store LOB snapshots
lob_snapshots = []
for t in range(T):
    bp, bs, ap, as_ = generate_lob_snapshot(mid_prices[t], rng)
    spread = ap[0] - bp[0]
    imbalance = (bs[0] - as_[0]) / (bs[0] + as_[0])
    lob_snapshots.append({
        'tick': t,
        'mid': mid_prices[t],
        'best_bid': bp[0], 'best_ask': ap[0],
        'spread': spread,
        'bid_depth': bs.sum(), 'ask_depth': as_.sum(),
        'obi': imbalance,                        # order book imbalance
        'bid_prices': bp, 'bid_sizes': bs,
        'ask_prices': ap, 'ask_sizes': as_
    })

lob_df = pd.DataFrame([{k: v for k, v in s.items() if not isinstance(v, np.ndarray)}
                        for s in lob_snapshots])

print(f'LOB statistics across {T} snapshots:')
print(f'  Avg spread : {lob_df["spread"].mean():.4f}  ({lob_df["spread"].mean()/lob_df["mid"].mean()*10_000:.2f} bps)')
print(f'  Avg bid depth  : {lob_df["bid_depth"].mean():.0f} shares')
print(f'  Avg ask depth  : {lob_df["ask_depth"].mean():.0f} shares')
print(f'  OBI range  : [{lob_df["obi"].min():.3f}, {lob_df["obi"].max():.3f}]')

In [ ]:
# ── Visualise: depth chart + spread + OBI ─────────────────────────────────────
SNAP_IDX = 500     # snapshot to display for depth chart
s = lob_snapshots[SNAP_IDX]

fig = ps.make_subplots(
    rows=2, cols=2, vertical_spacing=0.12, horizontal_spacing=0.10,
    subplot_titles=[
        f'LOB Depth Chart — tick {SNAP_IDX} (mid=${s["mid"]:.2f})',
        'Mid Price & Spread Over Time',
        'Bid vs Ask Depth Over Time',
        'Order Book Imbalance (OBI) — Short-term Price Predictor'
    ]
)

# Depth chart (horizontal bar)
fig.add_trace(go.Bar(
    y=[f'${p:.2f}' for p in s['bid_prices']], x=s['bid_sizes'],
    orientation='h', name='Bids', marker_color='#00d4aa', opacity=0.8), row=1, col=1)
fig.add_trace(go.Bar(
    y=[f'${p:.2f}' for p in s['ask_prices']], x=s['ask_sizes'],
    orientation='h', name='Asks', marker_color='#ff6b6b', opacity=0.8), row=1, col=1)

# Mid price + spread ribbon
ticks = lob_df['tick'].values
fig.add_trace(go.Scatter(x=ticks, y=lob_df['best_ask'], mode='lines',
    line=dict(color='#ff6b6b', width=0.8), fill=None, name='Ask'), row=1, col=2)
fig.add_trace(go.Scatter(x=ticks, y=lob_df['best_bid'], mode='lines',
    line=dict(color='#00d4aa', width=0.8),
    fill='tonexty', fillcolor='rgba(100,149,237,0.12)', name='Bid'), row=1, col=2)
fig.add_trace(go.Scatter(x=ticks, y=lob_df['mid'], mode='lines',
    line=dict(color='white', width=1.5), name='Mid'), row=1, col=2)

# Depth over time
fig.add_trace(go.Scatter(x=ticks, y=lob_df['bid_depth'], mode='lines',
    line=dict(color='#00d4aa', width=1.5), name='Bid depth', showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=ticks, y=lob_df['ask_depth'], mode='lines',
    line=dict(color='#ff6b6b', width=1.5), name='Ask depth', showlegend=False), row=2, col=1)

# OBI
obi = lob_df['obi']
fig.add_trace(go.Scatter(x=ticks, y=obi, mode='lines',
    line=dict(color='#f0c040', width=1.2), name='OBI', showlegend=False), row=2, col=2)
fig.add_hline(y=0, line_color='grey', line_dash='dot', row=2, col=2)

fig.update_layout(template='plotly_dark', height=620,
    title='Limit Order Book Dynamics')
fig.update_xaxes(title_text='Shares', row=1, col=1)
fig.update_xaxes(title_text='Tick', row=1, col=2)
fig.update_xaxes(title_text='Tick', row=2, col=1)
fig.update_xaxes(title_text='Tick', row=2, col=2)
fig.update_yaxes(title_text='Price Level', row=1, col=1)
fig.update_yaxes(title_text='Price ($)', row=1, col=2)
fig.update_yaxes(title_text='Shares', row=2, col=1)
fig.update_yaxes(title_text='OBI [-1, +1]', row=2, col=2)
fig.show()

# OBI predictive power: correlate OBI at t with next-tick mid return
obi_vals  = lob_df['obi'].values[:-1]
next_rets = np.diff(lob_df['mid'].values)
corr = np.corrcoef(obi_vals, next_rets)[0, 1]
print(f'OBI → next-tick mid return correlation: {corr:.4f}')
print('(Positive OBI → more bid pressure → price tends to rise)')

---
## Part 2 | Order Events & Lifecycle

An order in a production system moves through a strict **state machine**:

```
NEW → SUBMITTED → ACKNOWLEDGED → PARTIALLY_FILLED → FILLED
                              ↘ CANCELLED
                              ↘ REJECTED
```

Each transition emits an **event** that is pushed onto a FIFO queue. The event loop dequeues and routes each event to the appropriate handler — enforcing **causal ordering** (no future information ever touches current decisions).

In [ ]:
# ── Order state machine ────────────────────────────────────────────────────────
class OrderState(Enum):
    NEW            = auto()
    SUBMITTED      = auto()
    ACKNOWLEDGED   = auto()
    PARTIALLY_FILLED = auto()
    FILLED         = auto()
    CANCELLED      = auto()
    REJECTED       = auto()

class OrderSide(Enum):
    BUY  = 'BUY'
    SELL = 'SELL'

class OrderType(Enum):
    MARKET = 'MKT'
    LIMIT  = 'LMT'
    STOP   = 'STP'
    IOC    = 'IOC'   # Immediate-or-Cancel
    FOK    = 'FOK'   # Fill-or-Kill

# ── Event types ───────────────────────────────────────────────────────────────
class EvType(Enum):
    NEW_ORDER    = auto()
    ACK          = auto()
    PARTIAL_FILL = auto()
    FILL         = auto()
    CANCEL       = auto()
    REJECT       = auto()
    MARKET_DATA  = auto()

@dataclass
class Order:
    order_id:    int
    side:        OrderSide
    order_type:  OrderType
    qty:         int
    limit_price: Optional[float] = None
    stop_price:  Optional[float] = None
    state:       OrderState      = OrderState.NEW
    filled_qty:  int             = 0
    avg_fill_px: float           = 0.0
    events:      List            = field(default_factory=list)

    def fill(self, qty, price):
        """Apply a (partial) fill."""
        total_val = self.avg_fill_px * self.filled_qty + price * qty
        self.filled_qty  += qty
        self.avg_fill_px  = total_val / self.filled_qty
        self.state = (OrderState.FILLED if self.filled_qty >= self.qty
                      else OrderState.PARTIALLY_FILLED)

    @property
    def remaining(self):
        return self.qty - self.filled_qty

@dataclass
class Event:
    ev_type:  EvType
    tick:     int
    latency:  float     = 0.0   # simulated processing delay (ticks)
    order_id: int       = -1
    qty:      int       = 0
    price:    float     = 0.0
    reason:   str       = ''

print('Order state machine and event classes defined.')
print('States:', [s.name for s in OrderState])
print('Events:', [e.name for e in EvType])

In [ ]:
# ── Event queue with latency injection ────────────────────────────────────────
class EventQueue:
    """FIFO queue with optional latency: events are visible only after their delivery tick."""
    def __init__(self):
        self._pending: List[Event] = []

    def push(self, event: Event):
        self._pending.append(event)
        # keep sorted by (tick + latency) so we process causal order
        self._pending.sort(key=lambda e: e.tick + e.latency)

    def pop_ready(self, current_tick: int) -> List[Event]:
        """Return all events deliverable at or before current_tick."""
        ready, still_pending = [], []
        for e in self._pending:
            if e.tick + e.latency <= current_tick:
                ready.append(e)
            else:
                still_pending.append(e)
        self._pending = still_pending
        return ready

    def __len__(self):
        return len(self._pending)


# ── Simulate full order lifecycle for a single order ──────────────────────────
def simulate_order_lifecycle(order: Order, lob_snaps, start_tick,
                              network_lat=2, proc_lat=1, participation_cap=0.20):
    """
    Simulate an order from submission through fills/cancellation.
    Returns list of (tick, state, event_type) transitions.
    """
    eq = EventQueue()
    audit = []   # (tick, state_name, event_name, detail)

    # T0: strategy emits new order
    eq.push(Event(EvType.NEW_ORDER, start_tick, latency=0, order_id=order.order_id))

    for tick in range(start_tick, min(start_tick + 200, len(lob_snaps))):
        snap = lob_snaps[tick]
        events = eq.pop_ready(tick)

        for ev in events:
            if ev.ev_type == EvType.NEW_ORDER:
                order.state = OrderState.SUBMITTED
                audit.append((tick, 'SUBMITTED', 'NEW_ORDER', ''))
                # Exchange acknowledges after network + processing latency
                eq.push(Event(EvType.ACK, tick, latency=network_lat + proc_lat,
                              order_id=order.order_id))

            elif ev.ev_type == EvType.ACK:
                order.state = OrderState.ACKNOWLEDGED
                audit.append((tick, 'ACKNOWLEDGED', 'ACK', ''))

            elif ev.ev_type == EvType.PARTIAL_FILL:
                order.fill(ev.qty, ev.price)
                audit.append((tick, order.state.name, 'PARTIAL_FILL',
                              f'{ev.qty}sh @ ${ev.price:.2f}'))
                if order.state == OrderState.FILLED:
                    return audit

            elif ev.ev_type == EvType.REJECT:
                order.state = OrderState.REJECTED
                audit.append((tick, 'REJECTED', 'REJECT', ev.reason))
                return audit

        # Once acknowledged, try to match against the book each tick
        if order.state in (OrderState.ACKNOWLEDGED, OrderState.PARTIALLY_FILLED):
            if order.order_type == OrderType.MARKET:
                # Fill up to participation cap from the book
                if order.side == OrderSide.BUY:
                    avail_px  = snap['best_ask']
                    avail_qty = int(snap['ask_depth'] * participation_cap)
                else:
                    avail_px  = snap['best_bid']
                    avail_qty = int(snap['bid_depth'] * participation_cap)

                fill_qty = min(order.remaining, avail_qty)
                if fill_qty > 0:
                    eq.push(Event(EvType.PARTIAL_FILL, tick, latency=1,
                                  order_id=order.order_id,
                                  qty=fill_qty, price=avail_px))

            elif order.order_type == OrderType.LIMIT:
                if order.side == OrderSide.BUY and snap['best_ask'] <= order.limit_price:
                    fill_qty = min(order.remaining,
                                  int(snap['ask_depth'] * participation_cap))
                    if fill_qty > 0:
                        eq.push(Event(EvType.PARTIAL_FILL, tick, latency=1,
                                      order_id=order.order_id,
                                      qty=fill_qty, price=order.limit_price))
                elif order.side == OrderSide.SELL and snap['best_bid'] >= order.limit_price:
                    fill_qty = min(order.remaining,
                                  int(snap['bid_depth'] * participation_cap))
                    if fill_qty > 0:
                        eq.push(Event(EvType.PARTIAL_FILL, tick, latency=1,
                                      order_id=order.order_id,
                                      qty=fill_qty, price=order.limit_price))

        if order.state == OrderState.FILLED:
            break

    return audit


# ── Run two orders: market and limit ──────────────────────────────────────────
mkt_order = Order(order_id=1, side=OrderSide.BUY,
                  order_type=OrderType.MARKET, qty=500)
lmt_order = Order(order_id=2, side=OrderSide.BUY,
                  order_type=OrderType.LIMIT,  qty=500,
                  limit_price=mid_prices[100] - 0.05)

mkt_audit = simulate_order_lifecycle(mkt_order, lob_snapshots, start_tick=100)
lmt_audit = simulate_order_lifecycle(lmt_order, lob_snapshots, start_tick=100)

print('=== MARKET ORDER AUDIT TRAIL ===')
for tick, state, ev, detail in mkt_audit:
    print(f'  tick={tick:4d}  {state:<18}  {ev:<14}  {detail}')
print(f'  → avg fill px ${mkt_order.avg_fill_px:.4f}  total filled {mkt_order.filled_qty}')

print('\n=== LIMIT ORDER AUDIT TRAIL ===')
for tick, state, ev, detail in lmt_audit:
    print(f'  tick={tick:4d}  {state:<18}  {ev:<14}  {detail}')
print(f'  → avg fill px ${lmt_order.avg_fill_px:.4f}  total filled {lmt_order.filled_qty}')

In [ ]:
# ── Visualise order lifecycle on price chart ───────────────────────────────────
WINDOW = 60   # ticks around the orders
t0, t1 = 90, 90 + WINDOW
ticks_w = list(range(t0, t1))
mids_w  = mid_prices[t0:t1]
bids_w  = [lob_snapshots[t]['best_bid'] for t in ticks_w]
asks_w  = [lob_snapshots[t]['best_ask'] for t in ticks_w]

fig = go.Figure()
fig.add_trace(go.Scatter(x=ticks_w, y=asks_w, mode='lines',
    line=dict(color='#ff6b6b', width=0.8), name='Ask'))
fig.add_trace(go.Scatter(x=ticks_w, y=bids_w, mode='lines',
    line=dict(color='#00d4aa', width=0.8),
    fill='tonexty', fillcolor='rgba(100,149,237,0.10)', name='Bid'))
fig.add_trace(go.Scatter(x=ticks_w, y=mids_w, mode='lines',
    line=dict(color='white', width=1.5), name='Mid'))

# Mark fill events for market order
for tick, state, ev, detail in mkt_audit:
    if 'FILL' in ev and t0 <= tick < t1:
        fig.add_shape(type='line', xref='x', yref='paper',
            x0=tick, x1=tick, y0=0, y1=1,
            line=dict(color='#f0c040', dash='dot', width=1.5))
        fig.add_annotation(x=tick, xref='x', yref='paper', y=0.98,
            text=f'MKT fill<br>{detail}', showarrow=False,
            font=dict(color='#f0c040', size=9), xanchor='left')

# Limit price line
fig.add_hline(y=lmt_order.limit_price, line_color='cornflowerblue', line_dash='dash',
    annotation_text=f'Limit ${lmt_order.limit_price:.2f}', annotation_position='right')

# Mark fill events for limit order
for tick, state, ev, detail in lmt_audit:
    if 'FILL' in ev and t0 <= tick < t1:
        fig.add_shape(type='line', xref='x', yref='paper',
            x0=tick, x1=tick, y0=0, y1=1,
            line=dict(color='cornflowerblue', dash='dot', width=1.5))
        fig.add_annotation(x=tick, xref='x', yref='paper', y=0.85,
            text=f'LMT fill<br>{detail}', showarrow=False,
            font=dict(color='cornflowerblue', size=9), xanchor='left')

fig.update_layout(template='plotly_dark', height=420,
    title='Order Lifecycle: Market vs Limit Order Fills on LOB',
    xaxis_title='Tick', yaxis_title='Price ($)')
fig.show()

---
## Part 3 | Fills & Execution

### 3a — Aggressive vs Passive Execution

| Approach | Order type | Fills at | Cost | Risk |
|----------|-----------|----------|------|------|
| **Aggressive** | Market | Ask (buy) / Bid (sell) + slippage | High spread + impact | Low timing risk |
| **Passive** | Limit inside spread | Our limit price | Earn spread | Non-execution risk |

We simulate both on the same signal and compare: fill rate, average fill price, and P&L.

### 3b — Fill Probability Model

For a passive limit order at distance $\delta$ ticks from the best, the fill probability within $W$ ticks can be estimated from the empirical distribution of price excursions:

$$P(\text{fill} | \delta, W) = P\!\left(\min_{t \leq W} \Delta P_t \leq -\delta \cdot \text{tick}\right)$$

In [ ]:
# ── Strategy signal: OBI threshold ────────────────────────────────────────────
OBI_THRESH   = 0.30     # enter long when OBI > threshold (more bids than asks)
EXIT_TICKS   = 20       # hold for 20 ticks then exit
ORDER_QTY    = 300      # shares per trade
PARTICIPATION = 0.15    # max % of available depth per tick

obi_series = lob_df['obi'].values

signals = np.where(obi_series > OBI_THRESH)[0]   # entry ticks
# Avoid overlapping trades
entry_ticks = []
last_exit = -EXIT_TICKS
for t in signals:
    if t > last_exit + EXIT_TICKS:
        entry_ticks.append(t)
        last_exit = t

# Simulate aggressive vs passive for each entry
aggressive_trades, passive_trades = [], []

for entry in entry_ticks:
    if entry + EXIT_TICKS + 5 >= T:
        continue
    snap_entry = lob_snapshots[entry]
    mid_entry  = snap_entry['mid']

    # ── Aggressive: buy at ask + slippage ────────────────────────────────────
    slip_bps     = 2.0
    fill_px_agg  = snap_entry['best_ask'] * (1 + slip_bps / 10_000)
    fill_qty_agg = ORDER_QTY   # guaranteed full fill for market order

    # Exit at mid price after EXIT_TICKS
    exit_snap   = lob_snapshots[entry + EXIT_TICKS]
    exit_px_agg = exit_snap['best_bid']   # sell at bid on exit
    pnl_agg     = (exit_px_agg - fill_px_agg) * fill_qty_agg
    aggressive_trades.append({'entry': entry, 'fill_px': fill_px_agg,
                               'fill_qty': fill_qty_agg, 'exit_px': exit_px_agg,
                               'pnl': pnl_agg, 'type': 'Aggressive'})

    # ── Passive: limit at mid (earn the spread) ───────────────────────────────
    limit_px   = round(mid_entry, 2)    # post at mid; often filled when price ticks up
    filled_qty = 0
    fill_px_pass = 0.0
    for t_fill in range(entry + 1, min(entry + EXIT_TICKS, T)):
        snap_t = lob_snapshots[t_fill]
        if snap_t['best_ask'] <= limit_px:   # touched from above
            avail     = int(snap_t['ask_depth'] * PARTICIPATION)
            this_fill = min(ORDER_QTY - filled_qty, avail)
            if this_fill > 0:
                fill_px_pass = (fill_px_pass * filled_qty + limit_px * this_fill) / (filled_qty + this_fill)
                filled_qty  += this_fill
        if filled_qty >= ORDER_QTY:
            break

    exit_px_pass = exit_snap['best_bid']
    pnl_pass     = (exit_px_pass - fill_px_pass) * filled_qty if filled_qty > 0 else 0
    passive_trades.append({'entry': entry, 'fill_px': fill_px_pass,
                            'fill_qty': filled_qty, 'exit_px': exit_px_pass,
                            'pnl': pnl_pass, 'type': 'Passive'})

agg_df  = pd.DataFrame(aggressive_trades)
pass_df = pd.DataFrame(passive_trades)

print(f'Signals: {len(entry_ticks)}  |  OBI threshold: {OBI_THRESH}')
print(f'\nAggressive:  fill rate {(agg_df["fill_qty"]>0).mean():.0%}  '
      f'avg fill ${agg_df["fill_px"].mean():.4f}  total PnL ${agg_df["pnl"].sum():.2f}')
print(f'Passive:     fill rate {(pass_df["fill_qty"]>0).mean():.0%}  '
      f'avg fill ${pass_df["fill_px"].mean():.4f}  total PnL ${pass_df["pnl"].sum():.2f}')
print(f'\nPassive price improvement: {(agg_df["fill_px"].mean() - pass_df[pass_df["fill_qty"]>0]["fill_px"].mean())*10_000/mid_prices.mean():.2f} bps')

In [ ]:
# ── 3b: Fill probability model ─────────────────────────────────────────────────
# Empirically estimate P(fill | distance δ ticks, window W)
WINDOWS      = [5, 10, 20, 40]       # ticks to wait
DELTAS       = np.arange(0, 8)       # ticks below best ask

fill_prob = np.zeros((len(WINDOWS), len(DELTAS)))

for wi, W in enumerate(WINDOWS):
    for di, delta in enumerate(DELTAS):
        hits = 0
        trials = 0
        for t in range(50, T - W):
            limit_px = lob_snapshots[t]['best_ask'] - delta * TICK_SIZE
            # Check if price touches limit within W ticks
            future_asks = [lob_snapshots[t+j]['best_ask'] for j in range(1, W+1)]
            if any(a <= limit_px for a in future_asks):
                hits += 1
            trials += 1
        fill_prob[wi, di] = hits / trials if trials > 0 else 0

fig = ps.make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
    subplot_titles=['Fill Probability vs Limit Distance (ticks from best ask)',
                    'Partial Fill Distribution — Aggressive Orders'])

COLORS_W = ['#00d4aa', 'cornflowerblue', '#f0c040', '#ff6b6b']
for wi, (W, col) in enumerate(zip(WINDOWS, COLORS_W)):
    fig.add_trace(go.Scatter(
        x=DELTAS, y=fill_prob[wi],
        mode='lines+markers', name=f'W={W} ticks',
        marker=dict(size=7), line=dict(color=col, width=2)), row=1, col=1)

# Partial fill distribution
fill_fracs = agg_df['fill_qty'] / ORDER_QTY
fig.add_trace(go.Histogram(
    x=fill_fracs * 100, nbinsx=20, name='Fill fraction',
    marker_color='#00d4aa', opacity=0.8, showlegend=False), row=1, col=2)

fig.update_layout(template='plotly_dark', height=420,
    title='Fill Mechanics: Probability & Distribution')
fig.update_xaxes(title_text='Distance from best ask (ticks)', row=1, col=1)
fig.update_xaxes(title_text='Fill fraction (%)', row=1, col=2)
fig.update_yaxes(title_text='Fill probability', row=1, col=1)
fig.update_yaxes(title_text='Count', row=1, col=2)
fig.show()

print('Key insight: at δ=0 (posting at best ask), fill probability is high but')
print('degrades sharply for deeper limit levels — the trade-off of passive execution.')

---
## Part 4 | Order Management System (OMS)

An OMS is the central hub that:
1. **Routes orders** — validates and submits to the exchange
2. **Pre-trade risk checks** — position limits, capital limits, fat-finger
3. **Tracks positions** in real time (marks to market every tick)
4. **Reconciles** fills and maintains the order blotter

We implement a minimal but complete OMS and run it through the full tick series.

In [ ]:
# ── OMS Implementation ─────────────────────────────────────────────────────────
class RiskError(Exception):
    pass

class OMS:
    """
    Minimal Order Management System.
    Supports: order submission, pre-trade risk, position tracking, real-time PnL.
    """
    def __init__(self,
                 initial_cash      = 500_000,
                 max_position      = 2_000,      # shares, per instrument
                 max_order_size    = 1_000,
                 max_notional      = 200_000,    # $ per order
                 fat_finger_pct    = 0.05):       # reject if price > 5% from mid
        self.cash          = initial_cash
        self.max_position  = max_position
        self.max_order_sz  = max_order_size
        self.max_notional  = max_notional
        self.fat_finger    = fat_finger_pct

        self.position      = 0          # net shares (+ = long, - = short)
        self.avg_cost      = 0.0        # average cost basis
        self.orders: Dict[int, Order]   = {}    # order registry
        self.blotter       = []         # list of all fill events
        self.equity_curve  = []         # (tick, equity, position, cash)
        self.rejections    = []         # rejected orders log
        self._next_id      = 1

    # ── Pre-trade risk ─────────────────────────────────────────────────────────
    def _risk_check(self, order: Order, mid: float):
        """Raises RiskError on any violation."""
        notional = order.qty * (order.limit_price or mid)

        if order.qty > self.max_order_sz:
            raise RiskError(f'Fat finger: qty {order.qty} > max {self.max_order_sz}')

        if notional > self.max_notional:
            raise RiskError(f'Notional ${notional:,.0f} > limit ${self.max_notional:,.0f}')

        if order.limit_price and abs(order.limit_price - mid) / mid > self.fat_finger:
            raise RiskError(f'Price {order.limit_price:.2f} deviates >{self.fat_finger:.0%} from mid {mid:.2f}')

        sign = 1 if order.side == OrderSide.BUY else -1
        new_pos = self.position + sign * order.qty
        if abs(new_pos) > self.max_position:
            raise RiskError(f'Position {new_pos} > limit ±{self.max_position}')

    # ── Order submission ───────────────────────────────────────────────────────
    def submit(self, side: OrderSide, order_type: OrderType, qty: int,
               mid: float, limit_price=None, stop_price=None) -> Optional[Order]:
        order = Order(
            order_id    = self._next_id,
            side        = side,
            order_type  = order_type,
            qty         = qty,
            limit_price = limit_price,
            stop_price  = stop_price
        )
        self._next_id += 1
        try:
            self._risk_check(order, mid)
            self.orders[order.order_id] = order
            return order
        except RiskError as e:
            self.rejections.append({'reason': str(e), 'qty': qty, 'side': side.name})
            return None

    # ── Apply a fill ───────────────────────────────────────────────────────────
    def apply_fill(self, order: Order, fill_qty: int, fill_px: float, tick: int):
        sign = 1 if order.side == OrderSide.BUY else -1
        # Update average cost basis
        if sign == 1:   # buying
            total_cost     = self.avg_cost * self.position + fill_px * fill_qty
            new_pos        = self.position + fill_qty
            self.avg_cost  = total_cost / new_pos if new_pos != 0 else 0
        else:           # selling / short
            new_pos        = self.position - fill_qty
            if new_pos <= 0:
                self.avg_cost = fill_px if new_pos < 0 else 0

        self.position  = self.position + sign * fill_qty
        self.cash     -= sign * fill_qty * fill_px
        order.fill(fill_qty, fill_px)
        self.blotter.append({'tick': tick, 'order_id': order.order_id,
                              'side': order.side.name, 'fill_qty': fill_qty,
                              'fill_px': fill_px, 'position_after': self.position})

    # ── Mark to market ─────────────────────────────────────────────────────────
    def mark(self, tick: int, mid: float):
        equity = self.cash + self.position * mid
        self.equity_curve.append({'tick': tick, 'equity': equity,
                                   'cash': self.cash, 'position': self.position,
                                   'mid': mid})

print('OMS defined with pre-trade risk controls:')
print('  Max position: ±2,000 shares | Max order: 1,000 shares | Max notional: $200k')
print('  Fat-finger: reject if limit price > 5% from mid')

In [ ]:
# ── Run OMS through the tick series with OBI strategy ─────────────────────────
oms = OMS(initial_cash=500_000, max_position=2_000, max_order_sz=500)

HOLD    = 15
QTY     = 300
OBI_ENTER = 0.25
OBI_EXIT  = -0.20

pending: List[tuple] = []   # (order, fill_tick) — execute on next tick
last_entry = -HOLD

for tick in range(1, T):
    snap = lob_snapshots[tick]
    mid  = snap['mid']

    # Execute pending orders (next-tick rule)
    still_pending = []
    for order, sched_tick in pending:
        if tick >= sched_tick:
            fill_px  = snap['best_ask'] if order.side == OrderSide.BUY else snap['best_bid']
            fill_qty = min(order.remaining, int(snap['bid_depth' if order.side == OrderSide.SELL else 'ask_depth'] * 0.15))
            if fill_qty > 0:
                oms.apply_fill(order, fill_qty, fill_px, tick)
            if order.remaining > 0:
                still_pending.append((order, tick + 1))
        else:
            still_pending.append((order, sched_tick))
    pending = still_pending

    # Mark to market
    oms.mark(tick, mid)

    obi = lob_df.loc[tick, 'obi']

    # Entry signal: OBI above threshold, flat position
    if oms.position == 0 and obi > OBI_ENTER and tick > last_entry + HOLD:
        order = oms.submit(OrderSide.BUY, OrderType.MARKET, QTY, mid)
        if order:
            pending.append((order, tick + 1))
            last_entry = tick

    # Exit signal: OBI reverses or hold period elapsed
    elif oms.position > 0 and (obi < OBI_EXIT or tick >= last_entry + HOLD):
        order = oms.submit(OrderSide.SELL, OrderType.MARKET, oms.position, mid)
        if order:
            pending.append((order, tick + 1))

    # Stress-test: intentionally submit an oversized order to trigger risk check
    if tick == 400:
        oms.submit(OrderSide.BUY, OrderType.MARKET, 9999, mid)   # fat finger

eq_oms = pd.DataFrame(oms.equity_curve)
blotter = pd.DataFrame(oms.blotter)
n_trades = len(blotter[blotter['side'] == 'BUY'])

print(f'OMS simulation complete: {len(blotter)} fills  |  {n_trades} round-trips')
print(f'Final equity: ${eq_oms["equity"].iloc[-1]:,.2f}  (started $500,000)')
print(f'Risk rejections: {len(oms.rejections)}')
for r in oms.rejections[:3]:
    print(f'  → {r["reason"]}')

In [ ]:
# ── OMS dashboard ──────────────────────────────────────────────────────────────
fig = ps.make_subplots(rows=3, cols=1, vertical_spacing=0.08, shared_xaxes=True,
    subplot_titles=['Equity Curve', 'Position (shares)', 'Cash Balance ($)'])

ticks_eq = eq_oms['tick'].values

fig.add_trace(go.Scatter(x=ticks_eq, y=eq_oms['equity'], mode='lines',
    name='Equity', line=dict(color='#00d4aa', width=2)), row=1, col=1)
fig.add_hline(y=500_000, line_color='grey', line_dash='dot', row=1, col=1)

# Mark buy/sell fills
if len(blotter):
    buys  = blotter[blotter['side'] == 'BUY']
    sells = blotter[blotter['side'] == 'SELL']
    eq_idx = eq_oms.set_index('tick')['equity']
    for side_df, color, sym in [(buys, '#00d4aa', 'triangle-up'), (sells, '#ff6b6b', 'triangle-down')]:
        tks = side_df['tick'].values
        eqs = eq_idx.reindex(tks, method='nearest').values
        fig.add_trace(go.Scatter(x=tks, y=eqs, mode='markers',
            name=f'{side_df["side"].iloc[0] if len(side_df) else ""}',
            marker=dict(color=color, symbol=sym, size=8)), row=1, col=1)

fig.add_trace(go.Scatter(x=ticks_eq, y=eq_oms['position'], mode='lines',
    name='Position', line=dict(color='cornflowerblue', width=1.5),
    fill='tozeroy', fillcolor='rgba(100,149,237,0.15)'), row=2, col=1)
fig.add_hline(y=0, line_color='grey', line_dash='dot', row=2, col=1)

fig.add_trace(go.Scatter(x=ticks_eq, y=eq_oms['cash'], mode='lines',
    name='Cash', line=dict(color='#f0c040', width=1.5)), row=3, col=1)

fig.update_layout(template='plotly_dark', height=640,
    title='OMS Live Dashboard — OBI Strategy')
fig.update_yaxes(title_text='Equity ($)', row=1, col=1)
fig.update_yaxes(title_text='Shares', row=2, col=1)
fig.update_yaxes(title_text='Cash ($)', row=3, col=1)
fig.update_xaxes(title_text='Tick', row=3, col=1)
fig.show()

---
## Part 5 | Trade Management & P&L Attribution

P&L attribution **decomposes** total profit into three components:

$$\text{Total PnL} = \underbrace{P_{\text{exit}} - P_{\text{entry}}^{\text{mid}}}_{\text{Market Move}} - \underbrace{\frac{1}{2}(\text{Spread}_{\text{entry}} + \text{Spread}_{\text{exit}})}_{\text{Execution Cost}} + \underbrace{(P_{\text{mid}}^{\text{entry}} - P_{\text{fill}})}_{\text{Slippage Saved / Lost}}$$

By isolating these components we can distinguish **true alpha** (market move in our favour) from **execution quality** (how well we managed the fill).

In [ ]:
# ── Trade-level P&L attribution ────────────────────────────────────────────────
# Pair up buy and sell fills from the OMS blotter
if len(blotter) >= 2:
    trades_pnl = []
    buy_fills  = blotter[blotter['side'] == 'BUY'].reset_index(drop=True)
    sell_fills = blotter[blotter['side'] == 'SELL'].reset_index(drop=True)

    n_pairs = min(len(buy_fills), len(sell_fills))
    for i in range(n_pairs):
        b = buy_fills.iloc[i]
        s = sell_fills.iloc[i]
        bt, st = int(b['tick']), int(s['tick'])

        mid_entry  = lob_snapshots[bt]['mid']
        mid_exit   = lob_snapshots[st]['mid']
        sprd_entry = lob_snapshots[bt]['spread']
        sprd_exit  = lob_snapshots[st]['spread']

        market_move   = (mid_exit - mid_entry) * b['fill_qty']
        exec_cost     = 0.5 * (sprd_entry + sprd_exit) * b['fill_qty']   # spread paid
        slippage_comp = (mid_entry - b['fill_px']) * b['fill_qty']        # negative = we paid above mid
        gross_pnl     = (s['fill_px'] - b['fill_px']) * min(b['fill_qty'], s['fill_qty'])

        trades_pnl.append({
            'trade_id':    i + 1,
            'entry_tick':  bt,
            'exit_tick':   st,
            'hold_ticks':  st - bt,
            'fill_qty':    b['fill_qty'],
            'buy_px':      b['fill_px'],
            'sell_px':     s['fill_px'],
            'market_move': market_move,
            'exec_cost':   -exec_cost,
            'slippage':    slippage_comp,
            'gross_pnl':   gross_pnl
        })

    attr_df = pd.DataFrame(trades_pnl)
    print(f'Round-trip trade attribution ({n_pairs} trades):\n')
    print(attr_df[['trade_id','hold_ticks','fill_qty','gross_pnl',
                    'market_move','exec_cost','slippage']]
          .to_string(index=False, float_format='{:.2f}'.format))
    print(f'\nAggregated:')
    print(f'  Total Gross PnL  : ${attr_df["gross_pnl"].sum():>10,.2f}')
    print(f'  Market Move      : ${attr_df["market_move"].sum():>10,.2f}  ({attr_df["market_move"].sum()/attr_df["gross_pnl"].sum()*100:.1f}%)')
    print(f'  Execution Cost   : ${attr_df["exec_cost"].sum():>10,.2f}  ({attr_df["exec_cost"].sum()/attr_df["gross_pnl"].sum()*100:.1f}%)')
    print(f'  Slippage (net)   : ${attr_df["slippage"].sum():>10,.2f}  ({attr_df["slippage"].sum()/attr_df["gross_pnl"].sum()*100:.1f}%)')

In [ ]:
# ── P&L Attribution visualisation ─────────────────────────────────────────────
if len(blotter) >= 2 and len(attr_df) > 0:
    fig = ps.make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
        subplot_titles=['P&L Attribution — Stacked by Component per Trade',
                        'Cumulative P&L by Component'])

    trade_ids = attr_df['trade_id'].values

    fig.add_trace(go.Bar(x=trade_ids, y=attr_df['market_move'], name='Market Move',
        marker_color='#00d4aa', opacity=0.85), row=1, col=1)
    fig.add_trace(go.Bar(x=trade_ids, y=attr_df['exec_cost'], name='Execution Cost',
        marker_color='#ff6b6b', opacity=0.85), row=1, col=1)
    fig.add_trace(go.Bar(x=trade_ids, y=attr_df['slippage'], name='Slippage',
        marker_color='#f0c040', opacity=0.85), row=1, col=1)
    fig.add_hline(y=0, line_color='grey', line_dash='dot', row=1, col=1)

    fig.add_trace(go.Scatter(x=trade_ids, y=attr_df['market_move'].cumsum(),
        mode='lines+markers', name='Cum. Market Move',
        line=dict(color='#00d4aa', width=2), showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=trade_ids, y=attr_df['exec_cost'].cumsum(),
        mode='lines+markers', name='Cum. Exec Cost',
        line=dict(color='#ff6b6b', width=2), showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=trade_ids, y=attr_df['gross_pnl'].cumsum(),
        mode='lines+markers', name='Cum. Gross PnL',
        line=dict(color='white', width=2, dash='dot'), showlegend=False), row=1, col=2)
    fig.add_hline(y=0, line_color='grey', line_dash='dot', row=1, col=2)

    fig.update_layout(template='plotly_dark', height=430, barmode='relative',
        title='Trade-Level P&L Attribution')
    fig.update_xaxes(title_text='Trade ID', row=1, col=1)
    fig.update_xaxes(title_text='Trade ID', row=1, col=2)
    fig.update_yaxes(title_text='P&L ($)', row=1, col=1)
    fig.update_yaxes(title_text='Cumulative P&L ($)', row=1, col=2)
    fig.show()

In [ ]:
# ── Summary dashboard ──────────────────────────────────────────────────────────
fig = ps.make_subplots(
    rows=2, cols=3, vertical_spacing=0.14, horizontal_spacing=0.08,
    subplot_titles=[
        '1. LOB Depth Chart',
        '2. Order Lifecycle — Fill Ticks',
        '3. Fill Probability vs Distance',
        '4. OMS Equity Curve',
        '5. Aggressive vs Passive PnL',
        '5. P&L Attribution (Cumulative)'
    ]
)

# 1 — LOB depth
fig.add_trace(go.Bar(y=[f'${p:.2f}' for p in s['bid_prices']], x=s['bid_sizes'],
    orientation='h', marker_color='#00d4aa', opacity=0.8, showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(y=[f'${p:.2f}' for p in s['ask_prices']], x=s['ask_sizes'],
    orientation='h', marker_color='#ff6b6b', opacity=0.8, showlegend=False), row=1, col=1)

# 2 — order lifecycle: state transition ticks
state_labels = [a[1] for a in mkt_audit]
state_ticks  = [a[0] for a in mkt_audit]
state_colors = {'SUBMITTED': 'cornflowerblue', 'ACKNOWLEDGED': '#00d4aa',
                'PARTIALLY_FILLED': '#f0c040', 'FILLED': 'white'}
for st, tk in zip(state_labels, state_ticks):
    fig.add_trace(go.Scatter(
        x=[tk], y=[st], mode='markers',
        marker=dict(color=state_colors.get(st, 'grey'), size=10),
        showlegend=False), row=1, col=2)

# 3 — fill probability
for wi, (W, col) in enumerate(zip(WINDOWS[:2], ['#00d4aa', '#f0c040'])):
    fig.add_trace(go.Scatter(x=DELTAS, y=fill_prob[wi], mode='lines+markers',
        line=dict(color=col, width=2), showlegend=False), row=1, col=3)

# 4 — OMS equity
fig.add_trace(go.Scatter(x=eq_oms['tick'], y=eq_oms['equity'], mode='lines',
    line=dict(color='#00d4aa', width=1.5), showlegend=False), row=2, col=1)

# 5a — cumulative PnL comparison
if len(agg_df) and len(pass_df):
    fig.add_trace(go.Scatter(x=list(range(len(agg_df))),
        y=agg_df['pnl'].cumsum(), mode='lines',
        line=dict(color='#ff6b6b', width=2), showlegend=False), row=2, col=2)
    fig.add_trace(go.Scatter(x=list(range(len(pass_df))),
        y=pass_df['pnl'].cumsum(), mode='lines',
        line=dict(color='#00d4aa', width=2), showlegend=False), row=2, col=2)

# 5b — attribution
if len(blotter) >= 2 and len(attr_df) > 0:
    fig.add_trace(go.Scatter(x=trade_ids, y=attr_df['market_move'].cumsum(), mode='lines',
        line=dict(color='#00d4aa', width=1.5), showlegend=False), row=2, col=3)
    fig.add_trace(go.Scatter(x=trade_ids, y=attr_df['exec_cost'].cumsum(), mode='lines',
        line=dict(color='#ff6b6b', width=1.5), showlegend=False), row=2, col=3)
    fig.add_hline(y=0, line_color='grey', line_dash='dot', row=2, col=3)

fig.update_layout(template='plotly_dark', height=660,
    title='Day 18 Summary: Event-Driven Trading Engine')
fig.show()

---
## Summary — Production Trading Engine Checklist

| # | Component | Implemented in this notebook |
|---|-----------|------------------------------|
| 1 | Limit Order Book simulation | 5-level bid/ask with dynamic spread and depth |
| 2 | Order Book Imbalance (OBI) | Real-time OBI as a short-term price predictor |
| 3 | Order state machine | NEW → SUBMITTED → ACK → PARTIAL_FILL → FILLED / REJECTED |
| 4 | Event queue with latency | FIFO queue; orders visible only after network + processing delay |
| 5 | Aggressive execution | Market orders at ask/bid + slippage; guaranteed fill |
| 6 | Passive execution | Limit orders at mid; fill-at-touch with participation cap |
| 7 | Fill probability model | Empirical P(fill \| δ ticks, W window) across depths |
| 8 | OMS with pre-trade risk | Fat-finger, position limit, notional limit checks |
| 9 | Real-time position & equity | Cash + mark-to-market per tick; order blotter |
| 10 | P&L attribution | Market move vs execution cost vs slippage decomposition |

> *"Execution is itself a source of alpha. Efficient trading can save more than the signal earns."*